# FastViT → WHAM/HMR2 distillation (corrected)

This phase-two notebook resumes the spatial FastViT student trained on **person crops**, applies a stronger cosine/root-pose objective, evaluates on COCO val, and marks the checkpoint accepted only when every feature and pose-readout gate passes.

Before running: enable a Kaggle GPU (T4 or better), enable Internet, attach the COCO 2017 dataset, and attach the saved output of the first training notebook. The prior WHAM checkout, HMR2 checkpoint, and complete teacher cache are consumed directly from that read-only input; phase two writes only compact new artifacts under `/kaggle/working/fastvit_hmr2_phase2`.


In [ ]:
# Configuration
from pathlib import Path

SAVED_NOTEBOOK_ROOT = Path(
    '/kaggle/input/notebooks/nguyntrunglong/'
    'distill-fastvit-hmr2-kagglef9b9f724ae'
)
PHASE2_DIR = Path('/kaggle/working/fastvit_hmr2_phase2')
# Direct inner root for awsaf49/coco-2017-dataset: no recursive filesystem scan.
COCO_ROOT = Path('/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017')
DOWNLOAD_COCO = False

TRAIN_LIMIT = 48_000
VAL_LIMIT = 3_000
TEACHER_BATCH_SIZE = 8   # lower to 4 if HMR2 runs out of memory
TRAIN_BATCH_SIZE = 48    # lower to 24 if student training runs out of memory
WORKERS = 4
HEAD_EPOCHS = 0       # ignored when resuming
FINETUNE_EPOCHS = 6
TOKEN_LOSS_WEIGHT = 1.0
COSINE_LOSS_WEIGHT = 1.0      # raised after the validated run plateaued
POSE_LOSS_WEIGHT = 0.10
ROOT_POSE_LOSS_WEIGHT = 0.25  # root was the dominant pose error
FAIL_ON_REJECT = True

# A saved Kaggle notebook is mounted read-only. Locate its original run
# without copying the 2.7 GB teacher or token cache.
source_candidates = (
    SAVED_NOTEBOOK_ROOT / 'wham_fastvit_distill',
    SAVED_NOTEBOOK_ROOT,
)
SOURCE_RUN_DIR = next(
    (path for path in source_candidates
     if (path / 'fastvit_hmr2_best.pth').is_file()),
    None,
)
if SOURCE_RUN_DIR is None:
    visible = (sorted(path.name for path in SAVED_NOTEBOOK_ROOT.iterdir())
               if SAVED_NOTEBOOK_ROOT.is_dir() else [])
    raise FileNotFoundError(
        f'Could not find fastvit_hmr2_best.pth below {SAVED_NOTEBOOK_ROOT}. '
        f'Top-level entries: {visible}'
    )
RESUME_CHECKPOINT = SOURCE_RUN_DIR / 'fastvit_hmr2_best.pth'
SOURCE_HMR2_CHECKPOINT = SOURCE_RUN_DIR / 'checkpoints/hmr2a.ckpt'
SOURCE_TEACHER_CACHE = SOURCE_RUN_DIR / 'cache'
required_inputs = (
    RESUME_CHECKPOINT,
    SOURCE_HMR2_CHECKPOINT,
    SOURCE_TEACHER_CACHE / 'train_hmr2_tokens.npy',
    SOURCE_TEACHER_CACHE / 'train_hmr2_tokens.json',
    SOURCE_TEACHER_CACHE / 'val_hmr2_tokens.npy',
    SOURCE_TEACHER_CACHE / 'val_hmr2_tokens.json',
)
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError('Saved run is incomplete:\n' + '\n'.join(missing_inputs))
print(f'Reading prior run from: {SOURCE_RUN_DIR}')
print(f'Writing phase two to: {PHASE2_DIR}')


In [ ]:
# Pinned dependencies; Kaggle already supplies CUDA-enabled PyTorch.
%pip install -q timm==1.0.22 einops==0.8.1 yacs==0.1.8 gdown==5.2.0 pillow==11.3.0 tqdm==4.67.1


In [ ]:
# Materialize the reviewed training program embedded in this notebook.
import base64, gzip, hashlib

EXPECTED_SCRIPT_SHA256 = '9f8da0a840ea27b639ea100cdfd042b81e4a2a73f8b0050ed88532d6cbaabf3b'
SCRIPT_GZIP_BASE64 = 'H4sIAPERoGoC/919a3fbOJLod/0KDudDUx2KlpU4D01rzmby6M5s0p2TTs+cO15fhpYoixOJVJOUY7fX//3WAyAKICnLmZ49e25OEkkkUCgUCoWqQqHwxz8c7ary6DzLj9L80tte16sifzjwff9jmWS5l3ib4jxbp97rpKr/ln30qnq3SPPaqwuvTLdlsdjNU+/vPzx/903l/fDuw8TLNslFCq8/p3k0GHxcZZVXzctsW3vwbZFW2UWeLrxlUQLs/0wuLgD29+9/8fKiTs+L4nPone+gKJUuykWWJ+W1957Q8qpiUK8ANmKW5ReAwDzbpt48yb3zFH5dZumXdBF6l2lZZUWOX5N8AS+q3QbaLHZ1lS1S1Wo0eFN7y+wqrTwEWpTZBTS2bhCB/mzLLIcW4CF09HydbqrpYPCtd17UKy9P6y9F+blCJNLsMiUgVbJJoUkAkdSAwGiRlvBq4W0BIcB/XhbbPwEAKqroWKUKAao7OXl8dfxs4s3hVVp62CFvVwGE82uiLtZOvGqTrNdetYVGALdVmmAft+tkDqAu1sU5PEyABjgO26JYA6mo1TSZrwAoDU3lJSVSDp4AlTbJZ6Rnho2Wu20NDZa7vGLKJdBzqq/JjiNaA3Co+CUDUiD22zKl94hqMv98DtT3lmXxW5oDjcuqZsRX6XoxgmHwXvz04ifowBrGGUdoUXzJq7pMkw3z0LaocDyTBZa9SOrUS6E/14BznS2TeU2MlRJ66xpRgm7MP28L6ABiNy9yHnGg2qdP6dW2KOt4CQx8mdVxXpRAvuy3dBFtrz998mBcksG7ZB55Lwogybu32JdFNscBBD6s0vUSYe7y5DLJ1kgMrMI8BDzyNst3Vx6yAoxxhBNnMICOb7w4Xu7qXZnGMUwJxEAwRjUY6GflxTYpq1T/nleX+usqqVbr7Fz/5A/x4J/AUvr7JqlX+nsJFC02+le12tXZuvm1OwdWBj6pmifXzdc62zR41NfbtHmxK9fQblSmv+7SqtZPf8u2S5AM3NtFUifzdVJVwIK6axVSMTSvuCRwLXZLl3qPmNMLaJK4i58/z68bIuW7zRYGv/LyrcC16WJdlPOV9SPK82i5y2kMcTJU3mtu4/2bt7qBNyinVMtYRz/Pc/EwQuJVEXZBv38J398WCUztkL5XKfTx5905fKqKvy42UbIDAalRggeDAcrI+MVP7969+ejNPH9yfvJo+eTJsycPnx3Pnz16+uTx00dPz5+NT9LF0ycn58cnj+aPksmzE3+AMyL+/uWHN397Fb95iXWP//p4/bROr/9Rvh3/9vq3H1a/fX7xJF1+WP0y/sdfT75/9n/8AU6w+JcPb3+G8jcDD/74ND8n4+MnEYycP/X8VV1vq+nREUnsKpoX82LBHYqK8uIISlVHdqWQIV0m6/vCkVUUFDEdYmrmfmBF9aO9oMLB7eDNu+ffv/rx1cf43avnPwJJeHBrkINFGQTj6NHTk9CDj5PH9DF+PBwCu1erZJsGD0PvGP4ODZCfP77sgDGZPMPKk8kj/jjpgvH8w4sf3nx89eLjLx9e4VBqqVQlk0exkujxalNO4ssJSJLBfzSzJ2BxOvtY7tLhgB5572ld+ZDOYaWcElWJWFOQzyX9NISJs8UUBTw95uUlvpp6y3WRWM+u5TNcLvXvwWCRLnG5WsQkjOsVzNYAfxPcoTf6s/cjiH3Gg4VQhK+pzJCe5tuo+wXTcpPkO+i+8y5bqtfz3SKJsipuBHEw5MYMBCoiwMSwULaawQUqzRcVlgZBcZ7m89UmKT/DeCBxdU9XCazGMYq4AGXWlERVCEvNLv8cV7CAUL+h0lPvW+94PHmkPogSMACM2yID5sVSSpxHDDdghGj9ROhRsU3zwC/P/SGKK14MTe++rFAHo6a96Uy9jnCFDAw+ghqm5Wi3BRZKuRg3Ckv1rsz1+1V6xd8AJe55UhebbB7j+mL1HCbVDjoNktkZ7DpFQYeK2ow7g92Kq90S1CuCEPF374HnR/Vm6w/tatGXMqvTuE6v6gBbjRYg8KuA2guByKgozSZDrP5fuR96MGAFaIYXM39XL0dPW+CUMkRN606BPhPMiw3wBvDrOqvqUyDiGQznlwX3z/tv6hF0AT+cHqImWAf+A2jchz78EzQNDW0YwgTZVSs1MWnaNKtsJJqltmbwL2R1RVVg9EDnAD0gjb+skk2A2kS8yEpNeAB2CTPRRpQwxN9TPUuaYg0bANACx0S9QHFUrC9TxXzpukpbZXXb3pHn45LlNwWgBRAmVC5Kr4CCVeBwHHbWeoB/TltPSPpfZLVaCFqv5mvoXt9LvSpA/dXuHFaFzdF1sUryCsTRESIc9UOGEQ8Q/WH79Zn1ZNj8wrED8QzDU6U02Z2xtaqdcq+gB0k9QsGB30f4/9K/Eev/7f+9uWHAt7f+mY0M8gjiaD+talSFZ6Lxl6/+9uMvb9+2ioECf2cxZr/XCYy/eYGLFcoFmFowCWbe2B14mxTtkW96v0zr+Qq/sE2F3zawJvs826h3hsBWTUIMOkp0A+sJlj4EJEjXgqGQo3FFngURe7SBLqyroy0ZqPMjbZAc4boKSr8/7OBf0BrA5ngNg/ZjUb8udvniVVnCwr704TeYLoiDp/GbejfY3q2vBeplhganzR5UOIbS211t+KTpK1QakeqPP3549fylpE/ooTAkCTHgwQHezbZ64l5t0zkaaV/bXjdB9zcKhG76+YdZg4NLwQ+7HO0IJp7FI0vfoqG32cG6CHY7aP1oNoK+bM2R0LsAwt/oNm+NILJWMUTckaGkPhmL8PcTp8LKPFCoWjWkaDUvKmLLJIKfdUvYmmI9IlcUwL7Dx+YzNBHwj4oGExZMrBsXn8Ua1TTENsoFmuADe06TdQ1407sI/wNFcNGW8NliZtsoocdcOEOBazCElfLXXZbWLHl6BK610iAC01Z7HYzmk9tAUFtj6y1BV0wXfsPDgmAVaMYBMjlpT9533nE8Ho/1v72MvWw1SC4CPSnW1+yjATlhihhpQYxrXij21SjHaOkEC9DIspxUd+ZTRyMR7w8dcvS55ckGXoJBD2qV1xiJEahfG4u3ElCU0as1kw0B42J9l0lV2R4OZdVp6b9U3UMb/wYQuMX+3Ki6mjb6j+1xiOAn+pcyMDwC+B7qJk0l1N/TsoXu0o9uEOVbQA4MQxyb1hzjqndg/4qrE/I9OJMur5wi0T+yLa4lgUaUlPpiV87TNj/zc40hWiyiE3YbCte62M1Xjb6+BA05NpYe2DvYalwWRa1FHrnayCx05BwSB6i29NlFGX9Or1kqxTdU5zZChZxppqxvAlxBJSMJRIth10OSeMDVaJMr3Yz7tUlAVUgRGBjRMDgG5I09DGhE2IK2WVaAq6kNYGgLw1YpBIKlAo2ScBv4irdDT73EHy2pRGiABUpGoXl7K3rUMBV17G79wlkiXxS79YIg4Kh6zLweOeJBXOWLpFyw83SdXOMiep6uiy/ejaD1beT5FlD/57T2RiOk/4g6BystumwVscAOLEFoFWC9zYu8Vj5ePVoe0M0BJ6gWuYsy9H4Ndqzq/dD7s3dsSLBEz2vNegsYcp42pHCVIFPNGiUN4x76BZEV9Ip1nW3BXFbEw8Gqugg1RRxuGqxu/yu/m3BoIaZXME3BSq7Je97QsSqkh5f82g489mThfgUOMKg+m+wKFIgezUYR4HR8Juc5wUAt4l7T/P+3qUsdbeaqIqyvX/TOWyRb77S1GBcMoGNpG7sj8r+P2Vlcq+03PaezPZzvOwB/t5nQZui7ZF9L7iU9/bneIwp9Aa6jz74abaVsZdW8uEzLmJo5YC7VO6DyKb/C/8+mjoMVV9H+hTiUnKnkwGzPjLbKK/bjaqFoU/u2yAEc4+6DbN5BcOrhfgy6vUL04YHhxy2jlGSPMQsN4VNWtlFjEhlg0UVaBz4QsSy+LMCiHA9bswVr0rNGp4Ae59soqZKyTK4DF1hTDMCdnoGxsMBNqBnUIAf0w4lxpzdNjY5D76GYxmgr4g7dzDR6Og29yRlMzjH2oHkckc6P1hri9FtaFlUw1m2eF8W6kQuoACq4UbXbBEOY6TPvielv0znTpiofetOJ8SxtYGnd7DZoOzIK8CBIQO2cjaUie+UUSq5ahdT2MMhxDfOBrjgEITiJjPOGdrxn7MMPoKcITrcx0igN0XUdTbos2KvQu8Z93gUy/irNLlZoEm6SbUAgJTee+ufnxZV/1oGnGPYA3cAEjzEF+PBAQaYnXUPv9ge7YSHFXRifSCUM6mcVzLEM/d+EyjBC9XqIWo3S0lamCIKmV9TGd96jp9G4n6vVbzldDF/SzJqZ+WWWVmtHZoa8JQmYLYB8oUO/+GrGw8c/YflpF7m2ixzLItgd9Zq6GFqS8HyXrRcxi5BKIBNL1/8626AwBHRDz97yIVe6JIISjNvkmuzvGe1TR/i9BZ02MNjt33LpC0lJFMT5dYPkQlNVEQqwoB+o4VEh/4xWVXyKC4XC4VTrB2e3auSorwDwVIwuPpOrtxTtEpY0Gs6kLRk43hMSoNPZXuks/5h3UjJXpy0eoXdEgD6vtfmWVcToDdeeyd25D/TB+2NRtdotl2DWKOoYtQiHHsXn1CEWElB9O51SqbOB5b2hV3e4UX4svF1FQRUqRkaq0ED2G4dlHCeKakSx8mWyznCzS3Ezr6xVIDhJeAE1fh0cHGIwDiz31i7fw8nAccFssqpCW6nNRxG1ZnGT4ocsNzQTjZy5TomgwRdkogQ6dOzPhuYKm3tbnBhIY4jMu+0e2wmLwtiyU++GlF1uZXirSLRQ5pWrTH5JSwB7TpsmFBVx03ToFrf05usde4IUQBBqlodX7cH+ukOHrvJNYPhUIHe5KYQjov/1eE4tgQwDKUZwcvKYh1DUY3L5vv8zNUUhWuzJOV8n888w7ReIZ8hEQIzJiV0CC2Eo1SrJvfkaKIFv0gXGKVAEEAJdp8u6mSKRFuWw6KontMqYxbqGht3S172lq3lCao79Vgv539K24hjVMOsr1PENB1BRJpJYL5g8H3Xx6Pnr129+fBWKWtg4BjnA4o29VF/VY+iIAAYqGzHKjKF+4J9Arugvb94C2OcfTFlgpfW8WBflDFUx/OusVcxDdRGrqIs2H7CuTtv8H6mMUtNR9bD1kIzZpq1kEp0wAi2YEBLHpFKdnCjKK5oG3AhJdYpPCggstO9ZwSZYWQaOQE9k7MYL4DcVRRSoz6F28C69OEbNJI4DjEIL7yXDHFnFVv16GRm5MjPw7CIt8T4QCIEQUPhQC5ncCVSkQTkhAQ1lfVD1cXk2fcoX6ZVRJ9jMkgOIReozd/FBt4No4pTAGClKM5jZggIrnK47IrXPOUvCYNYlhZTLdl7kYEOC7fLh+7/4Qz0eQ5cgLtsiiKHqesMPuKHwEUMzX9EWbBnkefSuWOzWOq4DBMtP+fqao1WXy2yOwZ+0DbGhYlUTKfrpE+2AahYn3SrljYBPn4yA6uAwDD+IcUutc8+BYxuADsEwaiqa3v4RKE6bSYVGs/5SMIYmJhRXi0gVRKFZr8pid7ESQD59Qtc/7+F++uSlV+l8V0PvVJDxFkDhEoXNZxTIWSLe2XzlpfC8XF8LWLy7VXk/v3v/9mhdXOzKHYaS5tAstsqxsJ7yvKOmtyhS1piAltiFTdRA031gt9jM0OqObWdfsMOvu6wkl1BgQwMIKHHiRkanJW5Uh16r2GVW0xZ2axMG7BrbPW05mHTTbsRG7673G2AalNt16oxgUns3FlZyH0QNTqy2FXyKZ+EN2TY59EjOONpUcftH+B5IOC3owHvYrThGzQv9aRY6QimW5RleTBFKBrZh6+sqUtPoVBY4MxWEvW+PFElSFZwb8beYYQEdbyS028itK2iHYYBc7T4AoZY/HNgCvBmrmQCKJcVkpYJ2J1oc+NF8f5myULK4B9a9mKPJZ8d2hAk9BVG7cV/Qo/HkkfM03dar2WP7Ica1V7On9sPNehv3wMiA0aDK7LH7AqQtBs+gjmK9SDfnce9LDBOf+evkGsbIiZ4pcjRZGY3JU1FPLHRAtS9JuZAL99TSS3oUFRZbfGpinc1TNOB08BFMY3HQIlJNTEnS0tEMPEqgThAYsbVMEwpD3yRbvWhq/uAFAh1k/PfhZDp6OJH+G+4q+o8MFNCU0EucB5audBx6wkvDclXHyLJ3zQk7wIlJ3jxQ/Tk6FpSx9BL6zH6TiH9oDU09w+8dZq5aaF2mDgiPUPdjpj7B2P11l6a/AdaNpUEB3AGGBKTSUSpH6AxDR9JldmW8wj0Fp1JTNNshn1OMTdwUlynDCfhjOOXQSstehLIq4hLHnfDSu/RS9CNIeAlrHKo8GuCAtze4axRToA5/8CA4q3woYhEsrw+PgOZb/jUQapqrsoR99FAEsaJhuIAdUeLggVbXNl4XczJNZ/58u4MlcQMPVZDDF3L9VTEqHCKqREVhIs2gIQP01KdnMWKpfDe0ZkOhlvbV0GgYpTAMiupUvJk/hH1sYAaSi0LPb8qhboixVPNahGMwLClvXXBC6bLgVrAwk6yzajuNCEqwIlDCeoHeWNxuoqabR1bUQ/M0UjpDFV+UgFdA9LVQxygV5InGV8Q/I5ykuIPmYzi2b0BztVWyXgbaPuUjPjMxSXweVH/KA3gqertI5yhrIlXijG22QFib/nmWVPuq0vuuips0ybsqkpZ7XiyuY6zvVr21Nmyxe6HulJx9tLKq50HnZLtbmoDajsdOjHKtjle1D0zlabpgQ4DPd41YGuvzW8YA+F84H1sy8yvYIaIA4n+BLToA3Jc9bBC31kYdKug5WCocW3SXDd+cJChTsBfZehBB8qd8zEqBGXb7GtGjibGRCQxyNQv8EANCp9rJrmjuHFDQzUVsPgbDYcdRAWIwvbgoXTBoAotonQxVcD4VmbYErdpiDDudtNJd3+moZTYmJJxa5+ivMw5cfqiO6YknXUtcOHCiJ5oGDo60U9oSWV8zU58i0jgshY0ipliUg0WnzhkUFyBxq4NrmugswVPGc9PFaa0YYjHZKmhuk8TqAC0wvFDffVUfnqKHR0NrFYiZgaCYaFyUIqUP3p5aUOjojIiF90nVw3NgNKOOH/tS4mrbFNEf69VHUr0J56M4HIusHZF++r29W2XXunOvSu+RoyGuatLGNqhoFE/Cylxbt9MjodU7N8JXdBVH0YLtN29pD94gYm/iHhJRouIwkOE8WP8TOvM4XyUwhHxcWwVSeu9Af8VdUljs0DtwI8l+qw5cl7s88ns2pzAIt+nTn2cWM3UHX/5ScUia8keouC59npnmyNTBw4nNVCJOFpH4aFBgbNnersMJx+eRk8WI1ucuFBF6ZrwqfgeQ4KahzO3RjSTMrZ7Rwz6yqr6pIAtax2V3eZ1GX0A688sHftde//8kJfR2lTtuh/UOlimOViL3brxJsXPOzpbdeer3FzzAZXv8jx+HHkmkWdCWR107qtbpOCkhQu/m22/1ZMbDLM3UnHrjW+0bUMGCsw7/f7P6NY59vT5vVFTmTB00Dhb64HGJ8zNomgppNqm3QyUP1nRaGSqbo8uBsJwV8FA4OvXSOTNfRTgBbxS754fQGaTW15n6NC+3GQ1SUV7POm0EURKXnqqG9dWFhnvQMgDxPME+4eFq0xvuqoG2SKv5TEs10pSZH0WDdVEn6xlFgszTbG1xAay6hgJipVO+55mRY0eyZOjYXjpwK8tBVUtpXxuQn9rRIpXekKmMYRV6eZHH56BxY34G5/TG3fZWF2xhfTUbJaz8ZPkS9L18npKQcAPhVdaImZ7JzK3V0Ci82x38z1thwyipEKnAzDKBN257xF07ckweMUNB/D9+1BYCpwLEGZkuiNugb8UUpSn8aYgnSY9bUCM6xhn8izPdCHBLubZWHhXOqx6hTQEcn82V5rz/sPFk/Oip9MEAlTAUEvoWeua7shk71oT2WqCPz+I84ILsryMXqyMt9WhQ4Zg3xg6tMy921INxMy/IcUVmCkmxMUsvjfRQdl9wIylOTixhwxkMcqpAPxAQzvZFlXHnHyglraI4PzfozurzA2qef/BBZZgKXbW41w9YzVEF2YcCFiXxLrZ8xOU4hjEpswSmIfdQxeoFVuuquDcSSCA8XLbS0WM7jByeu9MRuh5y1bIOdHvDjlLN7qRK/4Mi9GfOXNO1P6mTBJFUSWguYICwlakmofDeMvlC8lj5Jvi82N6tSZNhZuphfKY6rX+/LUp3ewTTiERz0FZqlnprW4ew0jP4EoeZ+RrS0qeSn8zGHSoDatE5qFqu/x2lZaxZ18axSQWRUgwd0PpnPAuV40NnEyaPXhT55QQzAXAzwASTpyFYGSX84nX8OPTQxcFrtnP8GQB8Xxa77Y8YGoLqENTuKPLq7S9B+/HzRbKts8v0+eXFexgUwCIAAfBw2C75Wu0atN+8xU0Wbn2CuRQwkcLDjmJA7KSUZZSSFnaQXAUHXKAqUcbnoLEAS/ggFMBqismZE1qbEwTowOpVvWhqwyg2lRvONSmG2Ez/yl2gd6hQkKuPNJdllq4XXrHkrFCcEQKYUKmGaup9UzWz7aLMFgdtA+mdpIYV79wXktstklMD0crvtBlmaKnRblGXgzhc5ETFb9XGUDN8sDDIJ8gPjaQjXXFBLqr7BeZU07ZOH1oLf39QRROeYjQ1+6XjUeo0ZKlgM4hToQ7YuS2+NpxHaXz/YjRPh2tbabM2/t0mqFPG6DUtKnUqOY1GHHqx5ifulhtBZO9gijAvWywZFcTGTIHrii+bF6gh929iKvwIARMtVlTpB7WB4C68HQzJikbvRuZXLJprkr68FGlJTMre8aNHnUWVW55yZ1GfY6X+nGqX/ll3PVyoumqxj/4uGY1zmX3xmgqn7Lw/a06PHDPSw4F9wkzsjxHo7u2xQ7bIegQfjejBgk/KWKYM72YPtfBqeqo3F3Sc9GMUjCCC6zK7Csrm9M8B7W4TigvUdRqK4TEbHOsJLu3K21xSVqPXRhwHWPs0iiKKmpwi72eb2ehY7f6ASZ0vKJpGlzrGUvZLCU4MNL0deQG3+q16wgo3N4LqTrrF76wUfssYahyEQV6vMgofVAmjYPGvAlWWwdp4q2Hg0mBVzD8HbnGCODTVVDYK3PNL0UkWL1KwHVPVjD0QGspBw0P11cCKYbIHnApJmt9VQRFT9Ra0ND73LxsTcR5I6tFk6P2HDV1ZeFVGq0wQaEDRIksuMAkfjtPxbDQhMk2QTjR68AkDexyN1bkl3DhLNltqpmsEYJpNgJwq+jaBFgNuddhsSRX5MrtQhjVYlaTWTzsMmJCzWJpgDiMQf/cN8x6AUne6L3TjDMKls6ZAupnnw2pRc8996zCqOt3egYSxRuDNIu4XexhpCK9kvEngU1NV9DCizD+YQTXGqNjId/cxDuxMum668wfoDlLGd880/A3lOvufqWQTxAO6nN7lpoW/+qrRbwIzsrw77EMEhNxNe6cu0+v04ZnVQEdBQUtdUjM48nsMC8tmt41htpwvEqBDulXns5SnAB6ojU5PFTWPqMOkkExNL9Kt951dtH3++SqgltB/hhOWTuGJGjhnyQWhVgmxr8b1RlYDGoTA2CnhCoFxND6BxsfRsxMQ8eMI/w9AetD5R/TggjygL9sM3uD5So1CSEKmkRJE0jjdFvNVYIa4k1EGTcynCrCYSm0sFC72qUwOSs8LMI43GKKs5Ts9iH7Sj0N1mAMsDtDnnFLrMm7egJGM4/z2Q2iOfzTFQWCCCZ8sfqan+za2VRzGhbU1T7Gaa1gHY1bMVMbHUMj03tdEl76XGI4b95ZwQl7oqTII+iaymrTCB1nZ+9fYDm63yGhOnzrYesodaz0m1dF92HREvuGdaD7dUjV+TbEroTYj9CYEJ9S+oR7c+ujrTC7VLkp7q4At099rq0BDm+lvB9Zr2Je8JCSrAxqbIoZaqVNa7CVg+ts5TKXACfPFFmOyiMTWhTaSlKmld+XSHM8FLro3jIz55G4Z69zN8W94TJhkarfHoLK3iJUvAGsFml4jvRYYPwHJLPmwqhc9GFSUJUzg822rJsott4WOQGpka9KON2ouBRJu2OA+dCIHmqlrpfzQf1BujgCoKlfBUK+TMquvBXTDirZabLIGIspOXg9VPeXANx1ZruPfBPDOEeiqo3BwmtGSpZM0qvVQgnWCAizxtA8IucIen1mw1DMbZC+tW0IWmME8axV/0CF3oYZ42FHFlbRQoXnUUbxTOkMd+3nPfjwvQBF9BFhuSJoLWbutUrucvsRBI0+GTt7ePFeZtufrbMs6IU7a2MmvabTXAzR0TvzSqXA6OTePo3GnD1n1ERSRLsx15zi/rnyu12yqObT31T3eCXIlkF5GHsy42J5YcJsmAa95IbGe4zQP1NIXClZrFVHrYCiZq1VIOVQarmgVMCtk6DCQdM+7W8q4gJ9C786w37yRTN2MOPUn7tR+69ADVsOo4ilYYwg9NgGL640C9g3+/uYMMwExRafRo6VzaPwGWlQh96ZcB6EZpA7LwhjO/+jeKSeNEo2CHXLCv1Gd7FfqyBl0nlQpuoqaDTIVorFH0+JpIwwaiZn13EhtFYlpeRTxKBb7cpSo3l9IqjocNyEUnrby5P9Nnex31KapE7Ni9Bzr9+EBFTYZLW5liPbq3lq7o/Rqm2ASoCqwMNgXmNcLODhExxOn4wyEKNlu03wRtEDb0RotBdGupx921BFXZDSetKRjaWcQVhmLImXyJW68Rvs1Eb342nqIo++J8l3Km+pChwrHbwbdiuBXg+Jj/Oni0D5KXU73ExThbTUz2/rGr6iVJNvJaB9T1DOZcBItDcOu99z2UHhLrULOgY92rL4ZzlgFzDPvmOdKXbSihk2l7fikqcMc8+suyWs8zmpKUbaBEwuEQ2e7cedlBwZu9T1oOEU7cEFEyw2ZihIChVp0KvB6yIcWGGGz3AuaZQ5YEHkotQu4YZoGsuGlrkHCNf2gyqgXj886QNApiYNBHE8dGPoMhXUm5dqcV7ljxSVXjwj8d70gKneMkyeNl0q9xHJQdCuPmru+4sOeE4A+R62KAPuJoBAli63TOe4gwit5V4iMn+duiYNiUFb5k83ZMUl56jsUok8LErkqmRbiuaGFPxWEsY7DMC2wZfXVHqUKFul4BRpAUV5bl0iUxRetG9gUczcldRohLO84ItVmSLpe6MxQCC/Aopgca2ByS36h6DJ8Dvwk/OFNXcy6iyseaH5aA1RV1OlKQiLLRZXumzu+gGqSp19Qb5j5XXdUdF7rgfdeoByfV5fRSyDH3+lBwOVC0ejMbd9U59sz0FGeyuyW8iVSgMij3Z/ELjGoOSkLjWpelEpzvXsiGDWz+z1IHxhifbNV9CMivU2aczXC7+z7/vsyXWJcuMmHjll1ao825vAwBd3/hW6IVZpzkjtM3IfxY0qeN0FjJEA0cjHnrZqBgH5yQlYEPz/tlYNnets2m2dOklR0Uls6W+u4eICdjpL5PN2CnDRazUjQ87S1PjregyOvG0qXcfrgvki5uoiNWecKuhc9dyX8WhwlFu6idwZYyjad9/vQc4r+Luj1co6DJpVT+tg+HGW5fzuCHbPDxa2jiIuWpfkZ1I7HgBjMMj17TILO/fwvugoAjgHCXnawi7fK75/ZZpdwvYxrOsVorzju3VZPhtYR8Y4QWBEPygZp+6y4ZTJQPIs6aJvlJBSD0Tiiu8dOVDRjR2UyKrrqci2n6kEh/Rxw21iaenM9X+SY9ephiHkk6D8FFINbMbMPx+lSVAg647GwOSjDueNmKvESrIacGCn0AuCP0HuKmT3hK8CFfw/1cbX+LEsqEspKself+RQ5ckJ/H49t/LAaZ3Yl5HQfZG5Yk+JeqSG8dDm+CYzeMpHpx09k/LkLR+sQ9rw8HtO25DFOCwKH3aZHJ+JJNOYUs9jcd94TTkgLxdz4MN1Y3KS/uiut5I2TexwPJzwJnYcqf+/Uc6QKZ3Kdeqech+0Z/HsydrymImHvtE0UU/ZW6I2YruKf2wsrkbgaOreHbuZKVSw5rwKnKGe/G3lPMH4Er/EAW5kVFejEv0awp/cmmGbvx/S5j2SnMMxnwAonx920Qoh9xJI920MoUUwT6fFDh0gZSrKsvnZvNDw95hR8nAMPbbqOfTpOyrxNwXybYIiYhaRr1jUuCt1k2DQ+1AdzvgPmPxZXrg1ECvL1coRiG7RugL+YygSt0E/UEXH6h00AdKUjPZQXBx32lMuRzqjqqhu8WGRuZwcnxTXGpZrXiLYuq+Pl4CHq702B5+XFbgONvqc3QSsdfMkHBWatCi/TZbJb19UP6Xr7WheWjhgCGCWLBeJFVQRxRiM8JDdaZCU6+3GQdAYXgkq/Av/oM91ae4RlgRBHlONEH3FY4Bmk9dof3tGoL/LAH9JYlsOKoc9I9wDUV98QZADKydhmPpqPmJJql/p762NHRpisRSK0twaeoR8Zo+OwepLe6hjciA60MuGNN7WhycAks1pvZz7uItGNtM0pYD4P26S3l6lRzRladU02UzK8e3xIJxmR/qY7RnFAeoAePR2Px3shgBazp/7Du6pr4pAfeYTrcSecpwf04g4Qj/bDUEdHu6vuZxCkPjlMqq/AfQk6Yb3L030QDmh/3UxnlWy9GYF09OgwBPbDeZqOTg6Do+PS9sJ6vH9AcRNyhP7KkYoA7wZ0HI3vED9oQRwAaBxN9vcO16aD4Bzvxwhl4eHAxodJGAxoHmFIoSWnzOnUtox0xQ2fDqITRGCo7/JFktceQSS2tK4vyz3MdEwrorofiL1DmEgIFt07xY7EfJN8BoaB9tbp/RAGG6rw+Diel+BdWh6aPXjmyUJI4M3pJijJ5yGSEU28zsk4GU8ej5+N989qvJBukx68xrCVPwKjd9RssndzxLOTQwBpj8td0E7Gh0AzxvWIjOs+aE8Pwo2YX+l3PaAm4zvmNV7VBwsekPmfMM73VQMkA6J9DCBG5KK+55Shi4qIr3Q6eb5lhpYjvMec12HtkUyvMrzKCfS71Pv+/S8YurjbHsaKSp/d11HlaFEgpFqqHbkUGWz7MbAAHbQQpXVyGvQ+NQ4Q+wCa8ol0+dpdz4gCo6/Vbi62nHED+rdz7VPz+IAsSir2dF7obLsEuHngQNY9s65QFLdwCjjWJZz6XiuTL8K6g7GpZ13tZpDQN0rd/5Y3ZCi++YqNt8SogHhnpG7hNvJ+4TzE6qolOv5XrQBL926xc60mflN5L1BoYhJgLs/c1bqVSWfX+UCkRI2T0MIqzaVRBhMvijDE373qmuOom6hV+iWvcJi5Ny01EEPPp+I0AEO99dCAwu8HA4LCAozmBiUFYtq0G9hJhWgI4vcfXr1+++b7Hz7GP/2nzGJsm574R2Rca4XS3fRcY60RpB22UnBTz9XWRA4o3A3PvXtPQW2RvAe6uIDOqqkC0bor3fZgCvS+L57OeB6MpeGJw3HseKQvkQ/vvDFFiT0541Xwym6R4JRPLmGdwtDk9sS376R97r345eVzWhVgmjfJv9G/SAe6ZU4kmH6KATnCovHG8M+Ao56VXDQzl69T7bg6he/caDHH/mlsEv/blwK1wIQ8wfg57xN4zppwMI4OY/RhiMX68HNAKFzwqYsbHVyR25zuXTFd8sxKxtRTzxVdVh1NC6NHqNtUbPi35L+yx45h3nYQxAIKTU5REqkkIz0gDZIuQILo3JvN0et7b9RWlHVeD+2kidZOKvONyhdp8iq274fUQrxV1oqks5dzLOLLo553JJ0c7G3ITG49UWemlsheGzcJadq5XjspI1YhzmxX7bawpKFho91CVrq0P3nV52xLV8yQ3EhVJ1jV7ErkZq4mMGOIz9yBM0mO3b6GPb3T2Z+amqHLNjpttBQs4rx+f8ZQsQaGLXTMAzlBnaeK68XlaHosw24G7Epv1miwVgIzGWtrBNLh/cIVc0+vhPiwnv2P9gi3ORWcgQnNp2Uv3Wzra54agZIWYmOziTjDEy2zrsRWNhvcZ1+VkGfDW2eEOGR7VSRskId3DthcbdeEEk2mXMSDjw0C5upcuXraFx/mJN+QModryvAm6qeVjFkU7M6/3E66bElRrsrZQq0AsyEesJUxZl+XLzTny94IcQnfuxEtfyPffDP8Q9mb6JHHxs2EzqBOu2LfzjpSq3eMFSfANaTgmLgQxnDYqtOkP51Z1GuC3kLv5nZPInS5AmCiR5G6wxaqQ1FTRw4IqWlyRXbkpGlwbueR3KfDhI5EdgTa1zbZp/6ElpiUSzOrjvsyU1pE6MxOKZTQfYkqycHQmaeyWzKKZJV21Y7clBKCm6CSFNB9PRQU/1f6tycR57+3gzpUCLrXnH5R53TlHAgFJbR2ENpHLVp6rf8OXqvkcE28IUw8YYfrx2Fj3OkcXH/0nmuJJNRZTlsAFkhBd/fNwVJIy0tOd0FOcT7IjzseI9zyiLyPq1TBEwl8FykekYD21tfcCN1emdTqeNdFmeQ7DuyPTKCxHU54GshUAUoX0zstLLGq4VlLFeYu2fovgKIUCVoTxyTVCkDoHd4Mk17F6+okU1K31rvEpOypctG8uvRlxe64XhNZ07fs6Xb5VrX2AUqJl23AC3DGhojYs7gX7a5wOzTMbVRwW8RBrpUSw37fkVzcOh7tlKbY4XuHDXfQRscg6UDiDxwOzDWHw1bVc3j+mSdxWtV3jjgWirb1yt87jvZzMyCoaDTNmMdOZq8VngAlVWxiKz1NVUuvkVHqOkjbRajPM/OhJRxWCTpiRTShiguphFuQ8KBIaR0314qg1nqKwO0s9Iykwn51GWwWaI6b8bN8qRq/WBfnIEG1KmNpNjqTrdBZ5OUZaKaBVYk7+VO6PSp+8dO7d28+ijgjGuAmZz9/4SvkXOs1dOw06XTphrDPO+k7bptuCP1+Q9/SbNStBI62YzdmF7X0lB6oBqeuCxUOaWw/hB4caJtyatxWVjIL0KTF+RG2FGL9JI5l2WyzkUUx2Wl3SfQS4V0LKoyPdwJ0sJoKzfSujqPJnyj0DL+OT7xlsl5jXMCf7MtyhXD1OWYFIH/4/i90Nxn84/jQH0HNNHuQAAKWZZ5VdJGlvjte3WcmYYrj6lXLDd1k+2DlyT1278TjNUlA1PaSe+beKa5SgFBh9+S8U1SmDGFp1nXcvjvyz1KNoH4jP0TekcayM9aKCAScWle44eWWGJnZXlktAeH0QMoF51Xv7HfK9czwTmjOfOmA1FtCWI9ZRRc1p/bt7iqC1t6FM6d9LBq567tDZXOlByxqrYdNZUt7M1jd38TuWKuaO0o3KkEsDLzy2xzxrR2qP9OOSyYeeJioK/onQAoMXk5Sj4FM0U1XyJKyqG6iu5D9gO6pl9/N5B3xmveyXFxs15mbzUrqoy9+FGnIWiO55ewPB6RRoxQQduqH7hEyObychGO0/GJ2ZELjxifoKHUcJIGm61JPdXq3Lm/P9p76VneW7uloZ2ebBGGHdtTurN2h9l7mYT1srAjd1XAfpI6+dkNrCrYgnrVzEjVuMs6W9XyRbP4ecNe0UyxepPPkGkOuJq087pRtbEYnbIziwEYqZp1gpm7yj5ECLxKTYU2YQa2aYSuh2dGRd0y3iHdkCHG60JnwyxYJhZ05rMk7Q4UpiZtKADcTOOgMcDM7EVxnIrnWOAqYGo4tLPpTpzTdsxOUqc1Mk+FJ7EFpuRObwxfKUHWsBTTmKSc0ys5om5bLmHLiW2cgjRtJafNY3s38Zl0vSwKpPQmlN6Od20cwQPttz4hZfND1Kul8Lr32Dj0uOh4fogQ15e5Sf5qC+1UfY5fv13hsjrGPnn6FR8nN7fTFsoRaZ6GlTdXGv+9otP7z7bc3S6W3YDaZW3/ae72XxX9NMpkekNituwAaKu2B5oN0wlvTAU7QMUO8kZ5BmMHicetU4W2X/0VnBwHaDjvu6RKuOShheeXsrD+UwCC5TNvTr+Nsvxp9a7i0ViKOpjeqWEeUSK9XA71j5NXYx5T7DX6JQpeZL3OnssaGN7QvMcd2GhCIDk/RIXohDfLfzITZlsViN0/xbo28yEfcAiM9apDmzrSVwhau3OnvhF+ijaXls6DPjqxn/WPdM969YUH3ZYROQD0xRI2HKbyDNFZaA8stqSeJXd7y3DyYeceDxudm7/IZDO61x9e9W4bADt4r69zC4uSzZtH8OmFMJ/dth5Q5Acz3Q5pWOk4H4+2Ad52Id0+ut8F2H21vADht9B5r78rIYjfUf1z5u9lhZ9PtE8ztFvYfcO5oqfuEuX+eJsBMJlX+1zR0j8QK0m/BiJHmhjdlEotEvB2qk0vVK8w7X6wXvbzDqG+yxs9zDwbpq3vwwMeg4Tt1DxxQQ5MOGD1DJY5PXqoha07oAxDoMkCmtLRPTiSd/+i939UUOQw2UIYOP1iQ6QwoPpMnKCqQLus1fvJBETxMQXtf3guU7e/eKnjpFb5J8Xrh5a6CiZ14lxMJaZevMdcz+SM2Ce4YoksKOpBRJvhG8tnucUo1L5hPFNO8QoX0D/GemIde0jfxxrAQvTY/B86yhMVbOwxMgztuxrUS92hEpw2a4i2jOVWJTIR71iA5FSj2JeKxqOR6C1nH7/UZqlkPvRRniaZiU4ayw/cW7XbNG6odlhnIkPbO/aYm+bNiR3PNsbyUTwALFeTmcmN8wcGHshE+kxpzGgMGyOeU7sTIqhn9lm3trTA88hQzKNs31mDSszOotr3KTQ0TPzDFRcZAA4IPTtg7lqDFZ0s8G9BKDdqj1Ry0rRs61+caQvdl8yRKMCp7NkGdXT4uH8oBOzJgkCmHA6cqUZrCiVytEsPFxWhGuOEaV7slpgn1/eEw9HwcN5WgFNqamWa7fBh7R6Z1c+7vMNISeRCmWf45UNfk2teL067+XQzbGDcykp14FQ8W0msxww2qDfC9KLRNPmKRltWno5R/bp/dm3o3DYfeOkc3OqeThvWTPkWtzkWrXCaqOEA1ZBRwqfta+aBLCMn1mGRr0KpjPnPmxvP/fF2Bhf3qKnNOg/gfXv311YuPr17K9GhK3UWIuJ+1KKhJHmdeFIXsFfw2GGR4fxMyexyTDzqO8VhXHCs/NJ/xGvw/wfkANKu5AAA='
script_bytes = gzip.decompress(base64.b64decode(SCRIPT_GZIP_BASE64))
assert hashlib.sha256(script_bytes).hexdigest() == EXPECTED_SCRIPT_SHA256
SCRIPT_PATH = Path('/kaggle/working/distill_fastvit_hmr2.py')
SCRIPT_PATH.write_bytes(script_bytes)
print(f'Wrote {SCRIPT_PATH} ({len(script_bytes):,} bytes), SHA-256={EXPECTED_SCRIPT_SHA256}')


In [ ]:
# Cheap preflight: verifies crop padding, spatial output shape, target scaling, and metric math.
import subprocess, sys
subprocess.run([sys.executable, str(SCRIPT_PATH), '--self-test'], check=True)


In [ ]:
# Dataset preflight: resolves one unambiguous set of official train/val keypoint
# JSONs and image directories. This exits before parsing large JSONs or loading HMR2.
data_check = [
    sys.executable, str(SCRIPT_PATH),
    '--coco-root', str(COCO_ROOT),
    '--train-limit', '8', '--val-limit', '8',
    '--inspect-data',
]
subprocess.run(data_check, check=True)


In [ ]:
# Phase two: all large immutable inputs stay in the saved notebook mount.
command = [
    sys.executable, str(SCRIPT_PATH),
    '--work-dir', str(PHASE2_DIR),
    '--coco-root', str(COCO_ROOT),
    '--hmr2-checkpoint', str(SOURCE_HMR2_CHECKPOINT),
    '--teacher-cache-dir', str(SOURCE_TEACHER_CACHE),
    '--resume', str(RESUME_CHECKPOINT),
    '--train-limit', str(TRAIN_LIMIT),
    '--val-limit', str(VAL_LIMIT),
    '--teacher-batch-size', str(TEACHER_BATCH_SIZE),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--workers', str(WORKERS),
    '--head-epochs', str(HEAD_EPOCHS),
    '--finetune-epochs', str(FINETUNE_EPOCHS),
    '--token-loss-weight', str(TOKEN_LOSS_WEIGHT),
    '--cosine-loss-weight', str(COSINE_LOSS_WEIGHT),
    '--pose-loss-weight', str(POSE_LOSS_WEIGHT),
    '--root-pose-loss-weight', str(ROOT_POSE_LOSS_WEIGHT),
]
if DOWNLOAD_COCO:
    command.append('--download-coco')
if FAIL_ON_REJECT:
    command.append('--fail-on-reject')
print(' '.join(command))
training_result = subprocess.run(command, check=False)
if training_result.returncode:
    print('REJECTED OR FAILED: inspect the report below; do not deploy this checkpoint.')


In [ ]:
# Review the decision and expose only the three files we need.
import json
from IPython.display import FileLink, display

report_path = PHASE2_DIR / 'fastvit_hmr2_training_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
artifacts = (
    PHASE2_DIR / 'fastvit_hmr2_best.pth',
    PHASE2_DIR / 'fastvit_hmr2_training_report.json',
    PHASE2_DIR / 'fastvit_hmr2_history.csv',
)
for artifact in artifacts:
    if artifact.exists():
        display(FileLink(str(artifact)))
    else:
        print(f'Missing: {artifact}')


## After Kaggle

Download the checkpoint, report, and history linked above. Use `fastvit_hmr2_best.pth` only when `accepted` is `true` in the report. On the Mac, run `utils/export_fastvit_normalized.py`, then rerun the direct HMR2 comparison and the 3DPW/Core ML diagnostic before enabling the student in the iPhone app.
